# 02. RadixAttention 与 Prefix KV Cache

> 面向即将加入 SGLang 开源社区的开发者。建议从仓库根目录启动 Jupyter，
> 或者在 notebook 第一格运行路径检查。本文以本地源码为主，线上文档为索引。
> Markdown 中的源码摘录会插入 `# 注：...` 行内讲解；可执行代码 cell 则保持可运行。


In [ ]:
from pathlib import Path
import ast, inspect, re, textwrap

def find_repo_root(start=None):
    p = Path(start or Path.cwd()).resolve()
    for candidate in [p, *p.parents]:
        if (candidate / "python" / "sglang").exists() and (candidate / "docs").exists():
            return candidate
    raise RuntimeError("没有找到 SGLang 仓库根目录，请从仓库内启动 notebook。")

ROOT = find_repo_root()
print(ROOT)

def read_rel(path):
    return (ROOT / path).read_text()

def show_lines(path, start, end):
    lines = read_rel(path).splitlines()
    for i in range(start, min(end, len(lines)) + 1):
        print(f"{i:4d}: {lines[i-1]}")

def find_defs(path, names=None):
    tree = ast.parse(read_rel(path))
    rows = []
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)):
            if names is None or node.name in names:
                rows.append((node.lineno, type(node).__name__, node.name))
    return sorted(rows)


## 先纠正一个常见误解

`RadixAttention` 这个名字容易让人以为它只是一个 attention kernel。实际上它是一个跨层设计：

- `RadixAttention` layer 是模型中的 attention 抽象，负责把 Q/K/V 与 `ForwardBatch` 交给当前 attention backend。
- `RadixCache` / `HiRadixCache` 是 prefix KV cache 的 metadata tree，负责匹配、插入、拆分、锁定、驱逐。
- `SchedulePolicy` 会利用 prefix hit 长度调整 waiting queue 的优先级。
- KV memory pool 负责把 logical token span 映射到实际 KV slot。

RadixAttention 的收益来自“相同前缀不重新 prefill”，而不是仅仅来自某个单独 kernel。


In [ ]:
for path in [
    "python/sglang/srt/layers/radix_attention.py",
    "python/sglang/srt/mem_cache/radix_cache.py",
    "python/sglang/srt/managers/schedule_policy.py",
]:
    print("\n==", path)
    for row in find_defs(path, names={"RadixAttention", "RadixCache", "RadixKey", "TreeNode", "match_prefix_for_req", "SchedulePolicy"}):
        print(row)


## `RadixKey`：prefix matching 的基本单位

`RadixKey` 包装 token ids，同时带 `extra_key`。`extra_key` 很关键：同样 token 序列在不同 LoRA、cache salt、某些隔离策略下不能混用 KV。

它还支持 bigram view，这是 EAGLE 这类 speculative decoding 与 prefix cache 交互时会用到的特殊视图。


### `RadixKey` 的字段：token 序列之外还要带隔离维度

```python
# python/sglang/srt/mem_cache/radix_cache.py:35-75
  35: logger = logging.getLogger(__name__)
  36: 
  37: from sglang.srt.mem_cache.base_prefix_cache import (
  38:     BasePrefixCache,
  39:     DecLockRefParams,
  40:     DecLockRefResult,
  41:     EvictParams,
  42:     EvictResult,
  43:     IncLockRefResult,
  44:     InsertParams,
  45:     InsertResult,
  46:     MatchPrefixParams,
  47:     MatchResult,
  48: )
  49: from sglang.srt.mem_cache.events import KVCacheEventMixin
  50: from sglang.srt.mem_cache.session_radix_cache import SessionRadixCacheMixin
  51: from sglang.srt.mem_cache.utils import get_eviction_strategy, split_node_hash_value
  52: 
  53: if TYPE_CHECKING:
      # 注：类型检查分支：只给静态类型工具导入重依赖，运行时不会进入这条路径。
  54:     from sglang.srt.managers.schedule_batch import Req
  55: 
  56: 
  57: class RadixKey:
  58:     """is_bigram=True: token_ids holds raw tokens (N+1 for N bigrams); slices share one boundary token."""
  59: 
  60:     __slots__ = ("token_ids", "extra_key", "is_bigram", "limit")
      # 注：`RadixKey` 用 `__slots__` 固定 token_ids/extra_key/is_bigram/limit，减少 prefix-cache 热路径对象开销。
  61: 
  62:     def __init__(
  63:         self,
  64:         token_ids: array[int],
  65:         extra_key: Optional[str] = None,
  66:         is_bigram: bool = False,
  67:         limit: Optional[int] = None,
  68:     ):
  69:         # token ids sequence (raw ints in both modes)
  70:         self.token_ids = token_ids
      # 注：RadixKey 持有的 token 序列，是 radix tree 边上的实际匹配内容。
  71:         # extra key (e.g. lora_id, cache_salt)
  72:         self.extra_key = extra_key
      # 注：RadixKey 的隔离维度；LoRA、cache salt 或租户隔离不同就不能共享 KV。
  73:         # bigram view over token_ids: length = max(0, len(token_ids) - 1)
  74:         self.is_bigram = is_bigram
      # 注：标记当前 key 是否按 bigram 逻辑解释，主要服务 EAGLE/speculative 路径。
  75:         # Optional cap on raw tokens: behave as if token_ids were sliced to
```

**读这段时抓住：**

- `token_ids` 是 radix tree 的路径内容，但 `extra_key` 才决定这段 KV 能不能跨请求复用。
- `is_bigram` 让同一份 token array 可以被解释为 bigram 序列，避免为 EAGLE 等路径额外 materialize tuple list。
- `limit` 是 O(1) 视图上限，用来避免为了截断前缀而复制长 token 序列。


### `RadixKey.match`：长公共前缀用 galloping search 避免 Python 逐 token 扫

```python
# python/sglang/srt/mem_cache/radix_cache.py:120-160
 120:             raise ValueError("RadixKey slice step must be 1")
 121: 
 122:         if self.is_bigram:
      # 注：bigram 视图下一个逻辑 key 单元对应相邻 token 对，切片需要保留右边界 token 才能组成完整 bigram。
 123:             # bigrams [start, stop) span raw tokens [start, stop + 1);
 124:             # empty slice -> empty raw tokens (not a dangling boundary token).
 125:             raw = self.token_ids[start : stop + 1] if stop > start else array("q")
      # 注：bigram 切片需要包含 stop 右侧 token，才能让最后一个 bigram 单元完整。
 126:             return RadixKey(raw, self.extra_key, is_bigram=True)
 127:         return RadixKey(self.token_ids[start:stop], self.extra_key)
 128: 
 129:     def __repr__(self) -> str:
 130:         preview = self.token_ids[:10]
 131:         return f"RadixKey(extra_key={self.extra_key!r}, token_ids={preview}{'...' if len(self.token_ids) > 10 else ''}, is_bigram={self.is_bigram})"
 132: 
 133:     def page_aligned(self, page_size: int) -> RadixKey:
 134:         if page_size == 1:
      # 注：page size 为 1 时无需对齐，prefix hit 可以精确到任意 token。
 135:             return self
 136:         aligned_len = len(self) // page_size * page_size
      # 注：向下对齐到 page 边界后的可复用 token 数，避免返回半个 KV page。
 137:         return self[:aligned_len]
 138: 
 139:     def maybe_to_bigram_view(
 140:         self,
 141:         is_eagle: bool,
 142:         value: Optional[torch.Tensor] = None,
 143:     ) -> Tuple[RadixKey, Optional[torch.Tensor]]:
 144:         # O(1): flip the bigram flag instead of materializing a tuple list.
 145:         # value is paired with raw tokens and gets truncated to the bigram count.
 146:         if is_eagle and not self.is_bigram:
      # 注：EAGLE 使用 bigram prefix 语义；第一次进入该路径时把 key 视图切换为 bigram 而不复制 token 数组。
 147:             self.is_bigram = True
      # 注：标记当前 key 是否按 bigram 逻辑解释，主要服务 EAGLE/speculative 路径。
 148:             if value is not None:
      # 注：Radix key 变短或变成 bigram 后，KV index/value 必须裁到同样逻辑长度，防止 tree key 与 value 长度不一致。
 149:                 value = value[: len(self)]
      # 注：与 RadixKey 对应的 KV slot indices；tree 命中后 scheduler 会复用这些 device slots。
 150:         return self, value
 151: 
 152:     def _check_compatible(self, other: RadixKey) -> None:
 153:         if self.extra_key != other.extra_key:
      # 注：`extra_key` 不同代表 LoRA/cache salt/隔离域不同，即使 token 相同也不能复用同一段 KV。
 154:             raise ValueError(
 155:                 f"RadixKey operations require matching extra_key, but got "
 156:                 f"{self.extra_key=} != {other.extra_key=}"
 157:             )
 158: 
 159:     def match(self, other: RadixKey, page_size: int = 1) -> int:
 160:         """Logical-unit prefix length shared with ``other``. Result is rounded down to ``page_size``."""
```

**读这段时抓住：**

- 先检查 `extra_key`，这是 cache correctness 的第一道闸。
- 指数窗口 + 二分定位第一个分叉点，优化的是长 shared prefix 场景，这正是 prefix cache 的热点。
- `page_size` 对齐在这里完成；后续返回的 KV indices 才能安全对应 page allocator。


In [ ]:
from array import array
from sglang.srt.mem_cache.radix_cache import RadixKey

a = RadixKey(array("q", [1, 2, 3, 4, 5]), extra_key="tenant-a")
b = RadixKey(array("q", [1, 2, 3, 9]), extra_key="tenant-a")
c = RadixKey(array("q", [1, 2, 3, 4]), extra_key="tenant-b")

print("a vs b match:", a.match(b))
try:
    print(a.match(c))
except ValueError as e:
    print("extra_key isolation:", e)

bigram = RadixKey(array("q", [10, 11, 12, 13]), is_bigram=True)
print("bigram logical units:", list(bigram))
print("bigram len:", len(bigram))


## `RadixCache` 的核心操作

需要抓住四个操作：

- `match_prefix`：沿 radix tree 找最长已缓存前缀，返回 device indices、last node、host hit 等信息。
- `insert`：把完成 prefill/decode 的 KV span 插入 tree；必要时拆分节点。
- `inc_lock_ref` / `dec_lock_ref`：保护正在被请求使用的节点，避免被驱逐。
- `evict`：按策略回收未锁定叶子节点，释放 KV slots。


In [ ]:
for name in ["match_prefix", "insert", "cache_finished_req", "cache_unfinished_req", "evict", "inc_lock_ref", "dec_lock_ref", "_split_node"]:
    rows = find_defs("python/sglang/srt/mem_cache/radix_cache.py", {name})
    print(rows[0] if rows else "missing", name)


In [ ]:
show_lines("python/sglang/srt/mem_cache/radix_cache.py", 360, 455)


### `RadixCache.match_prefix`：返回的不只是长度，而是一组调度/加载锚点

```python
# python/sglang/srt/mem_cache/radix_cache.py:360-418
 360:     def match_prefix(self, params: MatchPrefixParams) -> MatchResult:
 361:         """Find the longest cached prefix of ``key`` in the radix tree.
 362: 
 363:         The logical namespace for prefix matching is determined by both the
 364:         token id sequence and the optional ``extra_key`` carried by ``RadixKey``.
 365:         Entries that share identical leading token ids but have *different*
 366:         ``extra_key`` values are intentionally kept disjoint and never share
 367:         prefix nodes. This is useful to:
 368: 
 369:         * Isolate KV cache lines for different LoRA / adapter IDs.
 370:         * Separate requests that intentionally should not share state (e.g.,
 371:           different sampling salt, cache version, or retrieval augmentation
 372:           context) by supplying a distinct ``extra_key``.
 373: 
 374:         Args:
 375:             params (MatchPrefixParams): Parameters containing the lookup key
 376:                 with a list of token ids and an optional ``extra_key`` namespace tag.
 377:                 If ``page_size > 1`` the length is internally truncated to a multiple
 378:                 of ``page_size`` before matching. Passing an empty key returns an
 379:                 empty result with the root as the last node.
 380: 
 381:         Returns:
 382:             MatchResult: ``device_indices`` is a 1-D ``torch.int64`` tensor of
 383:             the concatenated KV cache indices corresponding to the longest
 384:             cached prefix (may be length 0).
 385:             ``last_device_node`` and ``last_host_node`` (currently the same) are the tree node objects
 386:             representing the terminal node of the matched prefix. This method
 387:             may mutate internal structure by splitting an existing node if the
 388:             match ends inside a stored segment.
 389: 
 390:         Internal updates:
 391:             * Refreshes access metadata (timestamps) used by the
 392:                 configured eviction strategy.
 393:             * If the lookup ends inside a stored segment the node is split once
 394:                 to expose a precise boundary; this structural refinement improves
 395:                 subsequent match efficiency and does not duplicate data.
 396:         """
 397:         key = params.key
      # 注：prefix cache 查找/插入使用的 RadixKey；它同时包含 token ids 和 extra_key 隔离域。
 398:         key, _ = key.maybe_to_bigram_view(self.is_eagle)
      # 注：`key, _` 接收 `key.maybe_to_bigram_view` 的结果：EAGLE 路径会把 token 序列按 bigram 视图解释，prefix cache 需要用同一套 key 语义匹配 draft/target KV。
 399: 
 400:         if self.disable or len(key) == 0:
      # 注：prefix cache 被禁用或 key 为空时直接返回空命中，scheduler 会按无 cache 路径处理。
 401:             return self._empty_match_result
 402: 
 403:         key = key.page_aligned(self.page_size)
      # 注：prefix cache 查找/插入使用的 RadixKey；它同时包含 token ids 和 extra_key 隔离域。
 404: 
 405:         if len(key) == 0:
      # 注：page 对齐后可能没有完整 page 可复用，此时不能返回部分 page 的 KV index。
 406:             return self._empty_match_result
 407: 
 408:         value, last_node = self._match_prefix_helper(self.root_node, key)
      # 注：`value, last_node` 接收 `self._match_prefix_helper` 的结果：沿 radix tree 递归寻找最长已缓存前缀。
 409:         if value:
      # 注：radix helper 返回多个节点的 KV span；有命中时需要拼成连续 device index 交给 scheduler。
 410:             value = torch.cat(value)
      # 注：与 RadixKey 对应的 KV slot indices；tree 命中后 scheduler 会复用这些 device slots。
 411:         else:
 412:             value = self._empty_match_result.device_indices
      # 注：与 RadixKey 对应的 KV slot indices；tree 命中后 scheduler 会复用这些 device slots。
 413:         return MatchResult(
 414:             device_indices=value,
 415:             last_device_node=last_node,
 416:             last_host_node=last_node,
 417:             best_match_node=last_node,
 418:         )
```

**读这段时抓住：**

- `key.page_aligned` 说明 cache hit 的最小合法单位受 page size 约束。
- `value` 是已命中的 device KV slot indices；scheduler 后续会把它放进 `req.prefix_indices`。
- `last_device_node` / `last_host_node` / `best_match_node` 在普通 RadixCache 中基本相同，但 HiCache 会让它们分出 L1/L2/L3 语义。


### `RadixCache.insert` 与 `cache_finished_req`：什么时候把请求写入 prefix tree

```python
# python/sglang/srt/mem_cache/radix_cache.py:420-472
 420:     def insert(self, params: InsertParams) -> InsertResult:
 421:         if self.disable:
      # 注：禁用 radix cache 时 finished request 的 KV 不进入 tree，需要直接释放 KV slot。
 422:             return InsertResult(prefix_len=0)
 423: 
 424:         key = params.key
      # 注：prefix cache 查找/插入使用的 RadixKey；它同时包含 token ids 和 extra_key 隔离域。
 425:         value = params.value
      # 注：与 RadixKey 对应的 KV slot indices；tree 命中后 scheduler 会复用这些 device slots。
 426:         priority = params.priority
 427:         chunked = params.chunked
 428: 
 429:         key, value = key.maybe_to_bigram_view(self.is_eagle, value)
      # 注：`key, value` 接收 `key.maybe_to_bigram_view` 的结果：EAGLE 路径会把 token 序列按 bigram 视图解释，prefix cache 需要用同一套 key 语义匹配 draft/target KV。
 430:         key = key.page_aligned(self.page_size)
      # 注：prefix cache 查找/插入使用的 RadixKey；它同时包含 token ids 和 extra_key 隔离域。
 431:         if value is not None:
      # 注：Radix key 变短或变成 bigram 后，KV index/value 必须裁到同样逻辑长度，防止 tree key 与 value 长度不一致。
 432:             value = value[: len(key)]
      # 注：与 RadixKey 对应的 KV slot indices；tree 命中后 scheduler 会复用这些 device slots。
 433:         else:
 434:             # Debug/test fallback: use token ids themselves as values.
 435:             value = torch.tensor(key.token_ids[: len(key)], dtype=torch.int64)
      # 注：与 RadixKey 对应的 KV slot indices；tree 命中后 scheduler 会复用这些 device slots。
 436: 
 437:         prefix_len, last_node = self._insert_helper(
      # 注：insert helper 返回已有 prefix 长度和最终节点，调用方据此更新 cache metadata。
 438:             self.root_node, key, value, priority, chunked
 439:         )
 440:         return InsertResult(prefix_len=prefix_len, last_device_node=last_node)
 441: 
 442:     def cache_finished_req(self, req: Req, is_insert: bool = True):
 443:         """Cache request when it finishes."""
 444:         # In deterministic mode, disable finished request insertion to radix cache
 445:         if self.disable_finished_insert:
      # 注：确定性/调试模式可禁止 finished request 写入 radix tree，用于避免 cache 影响复现实验。
 446:             is_insert = False
 447: 
 448:         kv_committed_len = req.pop_committed_kv_cache()
      # 注：请求已经确认可提交的 KV 长度，只有这部分 KV 能进入 prefix cache 或被安全释放。
 449:         if self.disable:
      # 注：禁用 radix cache 时 finished request 的 KV 不进入 tree，需要直接释放 KV slot。
 450:             kv_indices = self.req_to_token_pool.req_to_token[
      # 注：req-to-token pool 中的物理 KV slot 索引，把请求 token 位置映射到实际 KV 存储。
 451:                 req.req_pool_idx, :kv_committed_len
 452:             ]
 453:             self.token_to_kv_pool_allocator.free(kv_indices)
 454:             return
 455: 
 456:         token_ids = (req.origin_input_ids + req.output_ids)[:kv_committed_len]
 457:         kv_indices = self.req_to_token_pool.req_to_token[
      # 注：req-to-token pool 中的物理 KV slot 索引，把请求 token 位置映射到实际 KV 存储。
 458:             req.req_pool_idx, : len(token_ids)
 459:         ]
 460: 
 461:         radix_key = RadixKey(
      # 注：finished request 写入 prefix tree 时使用的 key，包含 prompt+output token 和 extra_key 隔离域。
 462:             token_ids, req.extra_key, is_bigram=self.is_eagle
 463:         ).page_aligned(self.page_size)
 464:         key_len = len(radix_key)
 465:         values = kv_indices[:key_len].to(dtype=torch.int64, copy=True)
      # 注：写入 radix tree 的 KV slot index 副本；copy=True 避免后续 req pool 复用时污染 cache metadata。
 466: 
 467:         # Radix Cache takes one ref in memory pool
 468:         if is_insert:
      # 注：只有允许插入时才把 finished request 的 committed KV 提升为可复用 prefix cache。
 469:             priority = getattr(req, "priority", 0) or 0
 470:             result = self.insert(
      # 注：model worker forward/sampling 的输出，下一步必须交给 `process_batch_result` 更新请求和 cache 状态。
 471:                 InsertParams(key=radix_key, value=values, priority=priority)
 472:             )
```

**读这段时抓住：**

- `insert` 返回已有 prefix 长度和总长度，调用者据此知道新增了多少 cache metadata。
- `cache_finished_req` 使用 `origin_input_ids + output_ids[:-1]`，最后一个 token 通常还没有可作为未来 prefix 的完整 KV 后继语义。
- `kv_indices` 来自 req-to-token pool 的逻辑位置到 KV slot 映射，这里把请求生命周期内的 KV 提升为可复用 cache。


## Scheduler 如何“感知 cache”

`SchedulePolicy` 中的 LPM 表示 longest prefix match。等待队列里的请求会先计算 prefix hit，再按更可能复用 KV 的顺序进入 prefill。这就是为什么 prefix cache 不是被动缓存：调度器会主动把“能复用”的请求排到更合适的位置。


### `match_prefix_for_req`：scheduler 把 cache hit 写回 Req

```python
# python/sglang/srt/managers/schedule_policy.py:65-111
  65: 
  66: # Threshold for in-batch prefix cache.
  67: # If a request has a matched prefix length (against existing cache) less than this value,
  68: # the scheduler runs the in-batch prefix caching check for this request.
  69: # If we set it to -1, it means we disable in-batch prefix caching.
  70: IN_BATCH_PREFIX_CACHING_CHECK_THRESHOLD = int(
  71:     os.environ.get("IN_BATCH_PREFIX_CACHING_CHECK_THRESHOLD", "32")
  72: )
  73: 
  74: # Threshold for in-batch prefix cache.
  75: # If a request has a matched prefix length (within the waiting queue) larger than this value,
  76: # the scheduler deprioritizes this request
  77: IN_BATCH_PREFIX_CACHING_DEPRIORITIZE_THRESHOLD = int(
  78:     os.environ.get("IN_BATCH_PREFIX_CACHING_DEPRIORITIZE_THRESHOLD", "32")
  79: )
  80: 
  81: 
  82: IGNORE_EOS_RESERVE_TOKENS = 1
  83: 
  84: 
  85: def match_prefix_for_req(
      # 注：关键调用：`match_prefix_for_req` - 把 cache 命中结果写回 Req，供排序和 KV 分配使用。
  86:     tree_cache: BasePrefixCache,
  87:     req: Req,
  88:     token_ids: Optional[array[int]] = None,
  89:     *,
  90:     cow_mamba: bool = False,
  91:     include_req: bool = False,
  92: ):
  93:     if token_ids is None:
      # 注：调用方不传 token_ids 时，默认用 prompt + 已生成 token 作为 prefix matching key。
  94:         token_ids = req.origin_input_ids + req.output_ids
  95: 
  96:     match_result = tree_cache.match_prefix(
      # 注：prefix cache 返回的命中摘要，后续会展开到 `Req.prefix_indices/last_node/host_hit_length` 等字段。
  97:         MatchPrefixParams(
  98:             key=RadixKey(token_ids=token_ids, extra_key=req.extra_key),
  99:             cow_mamba=cow_mamba,
 100:             req=req if include_req else None,
 101:         )
 102:     )
 103:     if envs.SGLANG_RADIX_FORCE_MISS.get():
      # 注：强制 cache miss 的实验开关，用于对比 radix cache 对性能或正确性的影响。
 104:         match_result = zero_match_result(tree_cache, match_result)
      # 注：prefix cache 返回的命中摘要，后续会展开到 `Req.prefix_indices/last_node/host_hit_length` 等字段。
 105:     (
 106:         req.prefix_indices,
 107:         req.last_node,
 108:         req.last_host_node,
 109:         req.best_match_node,
 110:         req.host_hit_length,
 111:         req.swa_host_hit_length,
```

**读这段时抓住：**

- 这段是 RadixCache 和 Scheduler 的接缝：cache 返回 `MatchResult`，函数把它展开到 `Req` 字段。
- `num_matched_prefix_tokens` 把 device hit 和 host hit 合并成调度视角的 prefix hit。
- `SGLANG_RADIX_FORCE_MISS` 是调试开关，可以强制关闭命中以比较性能/正确性。


### `SchedulePolicy.calc_priority`：prefix cache 会反过来影响谁先 prefill

```python
# python/sglang/srt/managers/schedule_policy.py:145-214
 145:     RANDOM = "random"
 146:     ROUTING_KEY = "routing-key"  # prioritize by routing key frequency in running batch
 147: 
 148: 
 149: class SchedulePolicy:
 150:     Policy = Union[CacheAwarePolicy, CacheAgnosticPolicy]
 151: 
 152:     def __init__(
 153:         self,
 154:         policy: str,
 155:         tree_cache: BasePrefixCache,
 156:         enable_hierarchical_cache: bool,
 157:         enable_priority_scheduling: bool,
 158:         schedule_low_priority_values_first: bool,
 159:     ):
 160:         self.policy = self._validate_and_adjust_policy(policy, tree_cache)
      # 注：调度策略在初始化时会根据 cache 类型做校正，避免选择当前 prefix cache 不支持的策略。
 161:         self.tree_cache = tree_cache
      # 注：SchedulePolicy 持有 prefix cache 引用，排序前会用它计算 waiting request 的 prefix hit。
 162:         self.enable_hierarchical_cache = enable_hierarchical_cache
 163:         self.enable_priority_scheduling = enable_priority_scheduling
 164:         self.schedule_low_priority_values_first = schedule_low_priority_values_first
 165:         self.priority_sign = 1 if schedule_low_priority_values_first else -1
      # 注：priority 排序方向由 `schedule_low_priority_values_first` 决定，统一成乘法符号供 sort key 使用。
 166: 
 167:         # It is used to find the matching prefix for in-batch prefix caching.
 168:         self.waiting_queue_radix_tree = RadixCache.create_simulated()
      # 注：模拟 radix tree 用于 in-batch prefix caching，估计 waiting queue 内部请求之间的共享前缀。
 169: 
 170:     def calc_priority(
 171:         self, waiting_queue: List[Req], running_batch: Optional[ScheduleBatch] = None
 172:     ) -> None:
 173:         policy = self._determine_active_policy(waiting_queue)
      # 注：本轮实际使用的排序策略，队列长度或 cache 能力可能让它不同于用户配置的 policy。
 174: 
 175:         # Populate req.num_matched_prefix_tokens at schedule time. Cache-aware policies
 176:         # set it in _compute_prefix_matches; do the same full match for
 177:         # cache-agnostic policies when the radix supports it, so the load
 178:         # snapshot has it. Skip on decode (never prefills).
 179:         if (
 180:             not isinstance(policy, CacheAwarePolicy)
 181:             and self.tree_cache.supports_fast_match_prefix()
 182:             and get_global_server_args().disaggregation_mode != "decode"
 183:         ):
 184:             for r in waiting_queue:
 185:                 match_prefix_for_req(self.tree_cache, r)
      # 注：关键调用：`match_prefix_for_req` - 把 cache 命中结果写回 Req，供排序和 KV 分配使用。
 186: 
 187:         if self.policy == CacheAgnosticPolicy.FCFS:
      # 注：FCFS 不看 prefix cache；若开启 priority scheduling，只在到达顺序基础上加入用户优先级。
 188:             if self.enable_priority_scheduling:
 189:                 SchedulePolicy._sort_by_priority_and_fcfs(
 190:                     waiting_queue, self.priority_sign
 191:                 )
 192:             return
 193: 
 194:         if isinstance(policy, CacheAwarePolicy):
      # 注：cache-aware policy 会先把 waiting queue 的 prefix 命中写回 Req，再用 LPM/DFS 权重排序。
 195:             temporary_deprioritized = self._compute_prefix_matches(
 196:                 waiting_queue, policy
 197:             )
 198:             if policy == CacheAwarePolicy.LPM:
      # 注：LPM 策略优先最长 prefix hit 的请求，让即将进入 prefill 的 batch 更可能复用已有 KV。
 199:                 SchedulePolicy._sort_by_longest_prefix(
 200:                     waiting_queue, temporary_deprioritized
 201:                 )
 202:             elif policy == CacheAwarePolicy.DFS_WEIGHT:
      # 注：DFS weight 策略按 radix tree 局部性排序，目标是让相邻请求复用相近 cache 分支。
 203:                 SchedulePolicy._sort_by_dfs_weight(waiting_queue, self.tree_cache)
 204:             else:
 205:                 raise ValueError(f"Unknown CacheAware Policy: {policy=}")
 206:         else:
 207:             if policy == CacheAgnosticPolicy.FCFS:
 208:                 pass
 209:             elif policy == CacheAgnosticPolicy.LOF:
      # 注：LOF 关注输出长度而不是 cache hit，适合不想让 prefix cache 影响公平性的策略。
 210:                 SchedulePolicy._sort_by_longest_output(
 211:                     waiting_queue,
 212:                     self.enable_priority_scheduling,
 213:                     self.priority_sign,
 214:                 )
```

**读这段时抓住：**

- cache-aware policy 会先计算 prefix matches，再按 LPM 或 DFS weight 排序。
- 队列过长时 LPM 会退化到 FCFS，避免调度成本压过收益。
- 即使是 cache-agnostic policy，只要 radix 支持 fast match，也可能预先填充 prefix 信息供后续 load snapshot 使用。


In [ ]:
show_lines("python/sglang/srt/managers/schedule_policy.py", 65, 130)
show_lines("python/sglang/srt/managers/schedule_policy.py", 170, 235)


## `RadixAttention.forward`：模型层与 backend 的交界

模型文件中普遍会实例化 `RadixAttention`，但真正的 attention 实现由 `get_attn_backend().forward(...)` 决定。这样同一个模型层可以挂 FlashInfer、Triton、FA3、TRT-LLM、MLA 等 backend。


In [ ]:
show_lines("python/sglang/srt/layers/radix_attention.py", 75, 135)


## 小测试：page 对 prefix matching 的影响

KV cache 经常按 page 管理。`RadixKey.match(..., page_size=N)` 会把匹配长度向下对齐到 page 边界。这样能保证返回的 KV span 对底层 page allocator 是合法的。


In [ ]:
x = RadixKey(array("q", [1, 2, 3, 4, 5, 6]))
y = RadixKey(array("q", [1, 2, 3, 4, 9, 9]))
for page in [1, 2, 4]:
    print("page_size", page, "matched", x.match(y, page_size=page))


## 贡献者注意点

- 任何改变 prefix key 语义的 Feature，都要考虑 `extra_key`，否则可能跨租户/跨 LoRA 复用错误 KV。
- 任何改变 KV slot 生命周期的 Feature，都要检查 lock/ref、evict、cache_finished_req/cache_unfinished_req。
- 任何引入 page 粒度的 Feature，都要检查 `page_size > 1` 下是否仍然对齐。
